# Phase 3 — NumPy, 배열로서의 이미지 (예제)

이미지를 숫자 배열(ndarray)로 다루는 방법을 익힙니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

그래프 제목·축 라벨에 한글을 쓸 때 깨지지 않도록, 설치된 한글 폰트를 찾아 적용합니다. (없으면 라벨은 영어로 표기)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; print('한글 폰트 적용:', _n); break
else:
    print('한글 폰트를 찾지 못했습니다 — 라벨은 영어로 표기합니다.')
plt.rcParams['axes.unicode_minus'] = False

## 1. 이미지는 곧 숫자 배열
흑백 이미지의 각 픽셀은 밝기값(0=검정 ~ 255=흰색) 하나입니다. 작은 배열을 만들어 이미지처럼 표시해 봅니다.

In [ ]:
import numpy as np

img = np.array([
    [  0,  30,  60],
    [ 90, 150, 200],
    [220, 240, 255]], dtype=np.uint8)

print(img)
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('3x3 array as image'); plt.axis('off'); plt.show()

## 2. shape 와 dtype
`shape` 는 배열의 크기 (행, 열), `dtype` 은 값의 자료형입니다. 컬러 이미지는 (행, 열, 3) 으로 채널이 추가됩니다.

In [ ]:
print('shape :', img.shape)     # (3, 3)
print('dtype :', img.dtype)     # uint8
print('최댓값:', img.max(), '최솟값:', img.min())

color = np.zeros((3, 4, 3), dtype=np.uint8)   # 컬러 예시
print('컬러 shape :', color.shape)   # (3, 4, 3)

## 3. 변수에 이미지 담기 (단계별 보존)
가공 단계마다 변수를 나눠 담으면 중간 결과를 다시 보고 비교할 수 있습니다. 같은 변수에 덮어쓰면 원본이 사라지므로 주의합니다.

In [ ]:
img_raw  = img.copy()              # 원본 보존
img_bright = np.clip(img_raw + 40, 0, 255).astype(np.uint8)   # 밝게

print('원본 첫 행 :', img_raw[0])      # 원본은 그대로
print('가공 첫 행 :', img_bright[0])

# 주의: 아래처럼 같은 변수에 덮어쓰면 원본을 잃습니다
# img = np.clip(img + 40, 0, 255)   # img_raw 가 없으면 원본 비교 불가

## 4. 인덱싱 · 슬라이싱 = ROI 잘라내기
`img[행범위, 열범위]` 로 이미지의 일부(관심 영역, ROI)만 선택합니다. 번호는 0부터, `a:b` 는 a 이상 b 미만입니다.

In [ ]:
big = np.arange(36).reshape(6, 6).astype(np.uint8) * 7   # 6x6 예제
print(big)

roi = big[1:4, 1:4]      # 행 1~3, 열 1~3
print('\nROI:\n', roi)
print('ROI shape :', roi.shape)   # (3, 3)

big[1:4, 1:4] = 0        # 그 영역을 0으로 채우기
print('\n변경 후:\n', big)

## 5. 불리언 마스킹: 조건으로 픽셀 고르기
배열에 조건을 적용하면 각 픽셀이 True/False 인 마스크가 만들어집니다. 이것이 segmentation(영역 분리)의 기초입니다.

In [ ]:
img = np.array([
    [ 10, 160,  80],
    [200,  50, 255],
    [120, 240,  30]], dtype=np.uint8)

mask = img > 150          # True/False 배열
print('마스크:\n', mask)

print('조건 픽셀 개수 :', np.count_nonzero(mask))   # 밝기>150 픽셀 수
print('조건 픽셀 평균 :', img[mask].mean())

out = img.copy()
out[mask] = 255           # 조건 픽셀만 흰색으로
print('처리 결과:\n', out)

## 6. 집계 연산과 axis
배열 전체 또는 행·열 방향으로 평균·합 등을 계산합니다. `axis=0` 은 열별, `axis=1` 은 행별 결과이며, 생략하면 전체 하나의 값입니다.

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)

print('전체 평균 :', a.mean())          # 3.5
print('열별 평균 :', a.mean(axis=0))    # [2.5 3.5 4.5]
print('행별 평균 :', a.mean(axis=1))    # [2. 5.]
print('전체 표준편차 :', round(a.std(), 3))

## 6-1. 값과 위치: max vs argmax
`max`·`min` 은 **값**을 주고, `argmax`·`argmin` 은 **그 값이 있는 위치(인덱스)** 를 줍니다. 계측에서 "가장 밝은 곳", "변화가 가장 큰 곳"의 위치를 찾을 때 씁니다.

In [ ]:
a = np.array([12, 45, 7, 45])
print('max  =', a.max(),    '(값)')
print('argmax=', a.argmax(), '(위치: 첫 최댓값 인덱스)')
print('argmin=', a.argmin(), '(위치)')

# 2D: axis 로 방향 지정
m = np.array([[10, 90, 20],
              [80, 30, 95]])
print('열별 최댓값 위치(행 인덱스):', m.argmax(axis=0))   # [1 0 1]

계측 연결: 엣지는 밝기 변화(gradient)가 **최대인 위치**입니다. 그 위치를 `np.argmax` 로 찾습니다. (Phase 6 서브픽셀로 이어집니다.)

In [ ]:
prof = np.array([90, 92, 100, 160, 175, 178], float)
grad = np.abs(np.gradient(prof))
i = np.argmax(grad)          # gradient 가 최대인 '위치'
print('엣지 부근 정수 위치 i =', i, ' (grad 최댓값 =', round(grad[i],1), ')')

---
예제 실행을 마친 후 `03_practice.ipynb` 의 문제를 해결하십시오.